<a href="https://colab.research.google.com/github/hidrikoussay8-sketch/phishing_checker/blob/main/phishing_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1-charger le dataset

In [ ]:
!pip install ucimlrepo -q

from ucimlrepo import fetch_ucirepo
import pandas as pd

# Téléchargement du dataset (id 967 = PhiUSIIL Phishing URL)
dataset = fetch_ucirepo(id=967)
X = dataset.data.features
y = dataset.data.targets

# On regroupe features + label dans un seul DataFrame
df_full = pd.concat([X, y], axis=1)
print("Dimensions complètes :", df_full.shape)
print("Colonnes :", list(df_full.columns))

# On ne garde que ce dont on a besoin
df = df_full[["URL", "label"]].copy()
df.head()

Dimensions complètes : (235795, 55)
Colonnes : ['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']


,URL,label
0,https://www.southbankmosaics.com,1
1,https://www.uni-mainz.de,1
2,https://www.voicefmradio.co.uk,1
3,https://www.sfnmjournal.com,1
4,https://www.rewildingargentina.org,1


# 2-explorer les données et recoder le label

In [ ]:
# 1. Informations générales
df.info()

# 2. Valeurs manquantes
print("\nValeurs manquantes par colonne :")
print(df.isnull().sum())

# 3. Doublons
print("\nNombre d'URLs dupliquées :", df["URL"].duplicated().sum())

# 4. Équilibre des classes (label d'origine : 1 = légitime, 0 = phishing)
print("\nRépartition du label d'origine :")
print(df["label"].value_counts())
print(df["label"].value_counts(normalize=True).round(3))

# 5. Nouveau label : 1 = phishing, 0 = légitime
df["is_phishing"] = (df["label"] == 0).astype(int)

# 6. Vérification visuelle (on affiche seulement le texte, on ne visite rien)
print("\nExemples de phishing :")
print(df[df["is_phishing"] == 1]["URL"].head(3).to_string())
print("\nExemples légitimes :")
print(df[df["is_phishing"] == 0]["URL"].head(3).to_string())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 235795 entries, 0 to 235794
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   URL          235795 non-null  object
 1   label        235795 non-null  int64 
 2   is_phishing  235795 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 5.4+ MB

Valeurs manquantes par colonne :
URL            0
label          0
is_phishing    0
dtype: int64

Nombre d'URLs dupliquées : 425

Répartition du label d'origine :
label
1    134850
0    100945
Name: count, dtype: int64
label
1    0.572
0    0.428
Name: proportion, dtype: float64

Exemples de phishing :
11        http://www.teramill.com
20    http://www.f0519141.xsph.ru
21       http://www.shprakserf.gq

Exemples légitimes :
0    https://www.southbankmosaics.com
1            https://www.uni-mainz.de
2      https://www.voicefmradio.co.uk


# 3-nettoyer les URLs

In [ ]:
n_avant = len(df)

# 1. Garder uniquement les colonnes utiles
df = df[["URL", "is_phishing"]].copy()

# 2. Supprimer les espaces au début/fin et les URLs vides
df["URL"] = df["URL"].astype(str).str.strip()
df = df[df["URL"] != ""]

# 3. Repérer les URLs présentes avec des labels contradictoires
nb_labels = df.groupby("URL")["is_phishing"].nunique()
urls_conflit = nb_labels[nb_labels > 1].index
print("URLs avec labels contradictoires :", len(urls_conflit))
df = df[~df["URL"].isin(urls_conflit)]

# 4. Supprimer les doublons exacts
df = df.drop_duplicates(subset="URL").reset_index(drop=True)

# 5. Bilan
print("Lignes avant nettoyage :", n_avant)
print("Lignes après nettoyage :", len(df))
print("\nRépartition finale :")
print(df["is_phishing"].value_counts())
print(df["is_phishing"].value_counts(normalize=True).round(3))

URLs avec labels contradictoires : 0
Lignes avant nettoyage : 235370
Lignes après nettoyage : 235370

Répartition finale :
is_phishing
0    134850
1    100520
Name: count, dtype: int64
is_phishing
0    0.573
1    0.427
Name: proportion, dtype: float64


# 4-première famille de features (comptages sur l'URL entière)

In [ ]:
def extract_basic_features(url):
    url = str(url).strip()
    return {
        "url_length":   len(url),
        "n_dots":       url.count("."),
        "n_hyphens":    url.count("-"),
        "n_underscores": url.count("_"),
        "n_slashes":    url.count("/"),
        "n_at":         url.count("@"),
        "n_question":   url.count("?"),
        "n_equals":     url.count("="),
        "n_ampersand":  url.count("&"),
        "n_digits":     sum(c.isdigit() for c in url),
        "n_letters":    sum(c.isalpha() for c in url),
        "n_special":    sum(not c.isalnum() for c in url),
    }

tests = [
    "https://www.wikipedia.org/wiki/Machine_learning",
    "http://secure-login-verify.example-bank.com/update?id=123&user=abc@mail",
]
pd.DataFrame([extract_basic_features(u) for u in tests], index=["URL 1", "URL 2"]).T

,URL 1,URL 2
url_length,47,71
n_dots,2,2
n_hyphens,0,3
n_underscores,1,0
n_slashes,4,3
n_at,0,1
n_question,0,1
n_equals,0,2
n_ampersand,0,1
n_digits,0,3


# 5-features structurelles (domaine, chemin, HTTPS, mots suspects)

In [ ]:
from urllib.parse import urlparse
import ipaddress

SUSPICIOUS_WORDS = ["login", "signin", "verify", "account", "update",
                    "secure", "bank", "confirm", "password", "paypal", "wallet"]

def is_ip_address(host):
    try:
        ipaddress.ip_address(host)
        return 1
    except ValueError:
        return 0

def extract_structural_features(url):
    url = str(url).strip()
    # urlparse a besoin d'un schéma pour reconnaître le domaine
    url_to_parse = url if "://" in url else "http://" + url
    try:
        parsed = urlparse(url_to_parse)
        host = parsed.hostname or ""
    except ValueError:          # URL mal formée
        parsed = urlparse("")
        host = ""

    parts = host.split(".") if host else []
    return {
        "has_ip":            is_ip_address(host),
        "domain_length":     len(host),
        "n_subdomains":      max(len(parts) - 2, 0),
        "is_https":          int(url.lower().startswith("https://")),
        "path_length":       len(parsed.path),
        "query_length":      len(parsed.query),
        "n_suspicious_words": sum(url.lower().count(w) for w in SUSPICIOUS_WORDS),
    }

def extract_features(url):
    return {**extract_basic_features(url), **extract_structural_features(url)}

pd.DataFrame([extract_features(u) for u in tests], index=["URL 1", "URL 2"]).T

,URL 1,URL 2
url_length,47,71
n_dots,2,2
n_hyphens,0,3
n_underscores,1,0
n_slashes,4,3
n_at,0,1
n_question,0,1
n_equals,0,2
n_ampersand,0,1
n_digits,0,3


# 6-appliquer extract_features à tout le dataset

In [ ]:
# 1. Application de la fonction à chaque URL (peut prendre 30 à 90 secondes)
features_list = df["URL"].apply(extract_features)

# 2. Construction de X (features) et y (label)
X = pd.DataFrame(features_list.tolist())
y = df["is_phishing"]

# 3. Vérifications de base
print("Forme de X :", X.shape)
print("Forme de y :", y.shape)
print("Valeurs manquantes dans X :", X.isnull().sum().sum())

# 4. Moyenne de chaque feature selon la classe
comparaison = X.groupby(y).mean().T.round(2)
comparaison.columns = ["Légitime (0)", "Phishing (1)"]
comparaison

Forme de X : (235370, 19)
Forme de y : (235370,)
Valeurs manquantes dans X : 0


,Légitime (0),Phishing (1)
url_length,27.23,46.22
n_dots,2.16,2.38
n_hyphens,0.08,0.71
n_underscores,0.00,0.08
n_slashes,2.00,3.02
n_at,0.00,0.02
n_question,0.00,0.07
n_equals,0.00,0.15
n_ampersand,0.00,0.08
n_digits,0.05,4.34


# 7-diagnostic du biais

In [ ]:
resume = pd.DataFrame({
    "a_un_chemin": X["path_length"] > 0,
    "a_une_query": X["query_length"] > 0,
    "est_https":   X["is_https"] == 1,
}).groupby(y).mean().round(3)
resume.index = ["Légitime (0)", "Phishing (1)"]
print(resume)

print("\nExemples d'URLs légitimes :")
print(df[df["is_phishing"] == 0]["URL"].sample(8, random_state=42).to_string())

print("\nExemples d'URLs phishing :")
print(df[df["is_phishing"] == 1]["URL"].sample(8, random_state=42).to_string())

              a_un_chemin  a_une_query  est_https
Légitime (0)        0.000         0.00      1.000
Phishing (1)        0.597         0.06      0.486

Exemples d'URLs légitimes :
157484       https://www.atelierozmoz.be
189886            https://www.diemon.com
116574     https://www.wausauschools.org
42500          https://www.paademode.com
182408        https://www.boxturtles.com
225578         https://www.mmstadium.com
232854        https://www.brswimwear.com
86146     https://www.leathercouncil.org

Exemples d'URLs phishing :
8718      https://firebasestorage.googleapis.com/v0/b/hs...
69835     http://www.woodandthimble.com/www.labanquepost...
81505     https://mail.deliverylifesupport.com/public/0o...
114658                    http://www.windowofficeupdate.com
116998       https://osborne-legal-0utlook.firebaseapp.com/
125778                            http://www.alimahjour.com
12915      https://dev-ban-itaupyseguridad.pantheonsite.io/
124429                   https://celni-f7j.fi

# 8-charger un second dataset et l'inspecter

In [ ]:
# download this dataset "data.mendeley.com/datasets/vfszbj9b36"

from google.colab import files
import pandas as pd

uploaded = files.upload()
nom_fichier = list(uploaded.keys())[0]
print("Fichier chargé :", nom_fichier)

df2 = pd.read_csv(nom_fichier)
print("Forme :", df2.shape)
print("Colonnes :", list(df2.columns))
df2.head()

Saving URL dataset.csv to URL dataset.csv
Saving Phishing URLs.csv to Phishing URLs.csv
Fichier chargé : URL dataset.csv
Forme : (450176, 2)
Colonnes : ['url', 'type']


,url,type
0,https://www.google.com,legitimate
1,https://www.youtube.com,legitimate
2,https://www.facebook.com,legitimate
3,https://www.baidu.com,legitimate
4,https://www.wikipedia.org,legitimate


# 9-le nouveau dataset a-t-il des liens légitimes avec une suite ?

In [ ]:
from urllib.parse import urlparse

def a_une_suite(u):
    u = str(u).strip()
    if "://" not in u:
        u = "http://" + u
    try:
        p = urlparse(u)
        return int(len(p.path.strip("/")) > 0 or len(p.query) > 0)
    except ValueError:
        return 0

df2["a_une_suite"] = df2["url"].apply(a_une_suite)
df2["est_https"] = df2["url"].astype(str).str.lower().str.startswith("https://").astype(int)

print("Répartition des classes :")
print(df2["type"].value_counts())

print("\nProportion d'URLs avec une suite / en HTTPS, par classe :")
print(df2.groupby("type")[["a_une_suite", "est_https"]].mean().round(3))

for t in df2["type"].unique():
    print(f"\nExemples ({t}) :")
    print(df2[df2["type"] == t]["url"].sample(8, random_state=42).to_string())

# Second fichier
df3 = pd.read_csv("Phishing URLs.csv")
print("\nSecond fichier - forme :", df3.shape)
print("Colonnes :", list(df3.columns))
df3.head()

Répartition des classes :
type
legitimate    345738
phishing      104438
Name: count, dtype: int64

Proportion d'URLs avec une suite / en HTTPS, par classe :
            a_une_suite  est_https
type                              
legitimate        0.862      1.000
phishing          0.924      0.062

Exemples (legitimate) :
328491    https://www.yelp.com/biz/stone-mountain-clay-a...
60800                https://www.ivanhill5478.blogspot.com/
191230    https://www.facebook.com/people/Reggie-Walton/...
49392     https://www.freepages.genealogy.rootsweb.ances...
240956    https://www.manchester.edu/Athletics/basketbal...
343666                          https://www.isca-louis.com/
43523     https://www.espn.go.com/nfl/player/stats/_/id/...
34427                 https://www.en.wikipedia.org/wiki/Ico

Exemples (phishing) :
448480    http://educatingwellness.com/ServiceE/78c95/ho...
360722    http://grupodoma.com.mx/dropbox/mail.126.com/a...
390987    http://cke8q1.lucirabydevpps.info/zla4u55jxc

,url,Type
0,https://docs.google.com/presentation/d/e/2PACX...,Phishing
1,https://btttelecommunniccatiion.weeblysite.com/,Phishing
2,https://kq0hgp.webwave.dev/,Phishing
3,https://brittishtele1bt-69836.getresponsesite....,Phishing
4,https://bt-internet-105056.weeblysite.com/,Phishing


# 10-normaliser les URLs

In [ ]:
import re

def normaliser(u):
    u = re.sub(r"\s+", "", str(u))                          # retire espaces et retours à la ligne
    u = re.sub(r"^[a-zA-Z][a-zA-Z0-9+.\-]*://", "", u)       # retire http:// ou https://
    u = re.sub(r"^www\d*\.", "", u, flags=re.IGNORECASE)     # retire www.
    return u

d = df2[["url", "type"]].copy()
d["is_phishing"] = (d["type"] == "phishing").astype(int)
d["url_norm"] = d["url"].apply(normaliser)

# 1. Avant / après
print(d[["url", "url_norm"]].sample(6, random_state=1).to_string())

# 2. Doublons et conflits après normalisation
print("\nDoublons :", d["url_norm"].duplicated().sum())
nb_labels = d.groupby("url_norm")["is_phishing"].nunique()
print("URLs avec labels contradictoires :", (nb_labels > 1).sum())

# 3. Structure par classe, après normalisation
d["longueur"] = d["url_norm"].str.len()
d["a_une_suite"] = d["url_norm"].str.rstrip("/").str.contains("/").astype(int)
print("\nMoyennes par classe :")
print(d.groupby("type")[["longueur", "a_une_suite"]].mean().round(2))

                                                                                                                                                                                                                                                                              url                                                                                                                                                                                                                                                       url_norm
252042                                                                                                                                                                                                                        https://www.mp3raid.com/music/men_without_hats.html                                                                                                                                                                                                      

# 11-nettoyage final et vérification

In [ ]:
# 1. Vérification : aucune URL ne doit commencer par un schéma ou www
print("Préfixes restants :", d["url_norm"].str.contains(r"(?i)^(https?://|www\d*\.)").sum())

# 2. Retirer les URLs à labels contradictoires
nb_labels = d.groupby("url_norm")["is_phishing"].nunique()
conflits = nb_labels[nb_labels > 1].index
d = d[~d["url_norm"].isin(conflits)]

# 3. Supprimer les doublons
d = d.drop_duplicates("url_norm").reset_index(drop=True)

# 4. Bilan
print("Lignes restantes :", len(d))
print(d["is_phishing"].value_counts())

# 5. Aperçu lisible (texte uniquement, 80 premiers caractères)
for u, y in d.sample(6, random_state=1)[["url_norm", "is_phishing"]].values:
    print(y, repr(u[:80]))

/tmp/ipykernel_764/3947692461.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("Préfixes restants :", d["url_norm"].str.contains(r"(?i)^(https?://|www\d*\.)").sum())


Préfixes restants : 113
Lignes restantes : 443188
is_phishing
0    345729
1     97459
Name: count, dtype: int64
0 'last.fm/music/Bobby+Burns'
0 'web.me.com/sergehoude/SergeHoude/Reviews.html'
0 'stearmanworldflight.com/'
0 'docstoc.com/docs/302150/Turcotte-v-Fell-Case-Brief'
1 'ilcpowerbrand.com/images/placeholders/muah/'
0 'gunslot.com/pictures/michelle-manhart'


# 12-inspecter les 113 préfixes restants

In [ ]:
motif = r"(?i)^(?:https?://|www\d*\.)"
reste = d[d["url_norm"].str.contains(motif)]

print("Répartition par classe :")
print(reste["is_phishing"].value_counts())

print("\nExemples (texte uniquement) :")
for u, y in reste[["url_norm", "is_phishing"]].head(12).values:
    print(y, repr(u[:80]))

Répartition par classe :
is_phishing
1    112
0      1
Name: count, dtype: int64

Exemples (texte uniquement) :
0 'www.gov.uk'
1 'www1.taiwannation.com.tw/libraries/compat/cornell/login/ine...'
1 'http://www.oilchangeasheville.com/https:/www2.Santander.com.br/TopoEscolhaAcesso'
1 'http://leticiaaraujo.com.br/VAN-GOG.969/ATUALIZACAO.CLIENTE.SANTANDER.ATIVAR.CHA'
1 'http://leticiaaraujo.com.br/VAN-GOG.969/ATUALIZACAO.CLIENTE.SANTANDER.ATIVAR.CHA'
1 'http://www.ime.edu.co/firmasime/Santander/pessoa-fisica/index1.php?%20id=20,37,4'
1 'http://allatacadista.com.br/web/ipad/mesnos-riscos-online-site-seguro/index1.php'
1 'http://www.rainbowfarmsonline.com/Buttons/excel2013/index1.php?%20id=13,37,46,PM'
1 'http://www.ogrob.org/240972,81/index22.html'
1 'http://186.67.93.102/signin/de/cgi-bin/?webscr&amp;cmd=_reset-run&amp;SESSION=&l'
1 'https://thoaimedia.com/save/index.php?error=Cannot%20login:%20[-2]%20You%20have%'
1 'http://sgademexico.com/tmp/Inc,Dropbox/1/view.php'


# 13-normalisation répétée, puis nettoyage

In [ ]:
motif = re.compile(r"^(?:[a-zA-Z][a-zA-Z0-9+.\-]*:/{1,2}|www\d*\.)", re.IGNORECASE)

def normaliser(u):
    u = re.sub(r"\s+", "", str(u))
    while motif.match(u):                # on répète tant qu'un préfixe reste
        u = motif.sub("", u, count=1)
    return u

d["url_norm"] = d["url"].apply(normaliser)
d = d[d["url_norm"] != ""]

# Conflits, puis doublons
nb_labels = d.groupby("url_norm")["is_phishing"].nunique()
d = d[~d["url_norm"].isin(nb_labels[nb_labels > 1].index)]
d = d.drop_duplicates("url_norm").reset_index(drop=True)

print("Préfixes restants :", d["url_norm"].str.contains(motif).sum())
print("Lignes :", len(d))
print(d["is_phishing"].value_counts())

Préfixes restants : 0
Lignes : 443148
is_phishing
0    345729
1     97419
Name: count, dtype: int64


# 14-analyser df3 (Phishing URLs.csv)

In [ ]:
d3 = df3[["url", "Type"]].copy()
print("Types présents :")
print(d3["Type"].value_counts())

d3["url_norm"] = d3["url"].apply(normaliser)
d3 = d3[d3["url_norm"] != ""]
print("\nDoublons internes :", d3["url_norm"].duplicated().sum())
d3 = d3.drop_duplicates("url_norm")

phish_d = set(d.loc[d["is_phishing"] == 1, "url_norm"])
legit_d = set(d.loc[d["is_phishing"] == 0, "url_norm"])
u3 = set(d3["url_norm"])

print("\nURLs uniques dans df3 :", len(u3))
print("Déjà dans le phishing de d :", len(u3 & phish_d))
print("Présentes dans le légitime de d (contradiction) :", len(u3 & legit_d))
print("Nouvelles URLs :", len(u3 - phish_d - legit_d))

Types présents :
Type
Phishing    54807
Name: count, dtype: int64

Doublons internes : 509

URLs uniques dans df3 : 54298
Déjà dans le phishing de d : 48
Présentes dans le légitime de d (contradiction) : 0
Nouvelles URLs : 54250


# 15-comparer le phishing de d et de df3

In [ ]:
p1 = d[d["is_phishing"] == 1]["url_norm"]
p3 = d3["url_norm"]

def profil(s):
    return {
        "longueur moyenne": s.str.len().mean(),
        "% avec chemin": s.str.rstrip("/").str.contains("/").mean() * 100,
        "% .php": s.str.contains(r"\.php", case=False).mean() * 100,
        "% IP en domaine": s.str.match(r"^\d{1,3}(?:\.\d{1,3}){3}").mean() * 100,
        "% mots suspects": s.str.contains(
            r"login|signin|verify|account|update|secure|bank|confirm|password|paypal|wallet",
            case=False).mean() * 100,
    }

print(pd.DataFrame({"phishing de d": profil(p1), "phishing de df3": profil(p3)}).round(1))

                  phishing de d  phishing de df3
longueur moyenne           58.7             58.5
% avec chemin              92.1             54.4
% .php                     25.0              3.1
% IP en domaine             4.1              0.9
% mots suspects            17.5              5.7


# 16-mettre df3 de côté

In [ ]:
# Test externe : phishing de df3 absent de d
test_ext = d3[~d3["url_norm"].isin(set(d["url_norm"]))].copy()
test_ext["is_phishing"] = 1
test_ext = test_ext[["url_norm", "is_phishing"]].reset_index(drop=True)

print("Test externe :", len(test_ext), "URLs de phishing")
print("Intersection avec d :", len(set(test_ext["url_norm"]) & set(d["url_norm"])))

# Domaine (partie avant le premier "/") pour repérer les fuites par domaine
dom = lambda s: s.str.split("/").str[0].str.lower()
partage = set(dom(test_ext["url_norm"])) & set(dom(d["url_norm"]))
print("Domaines communs entre d et le test externe :", len(partage))

Test externe : 54250 URLs de phishing
Intersection avec d : 0
Domaines communs entre d et le test externe : 77


# 17-retirer les domaines communs du test externe

In [ ]:
dom = lambda s: s.str.split("/").str[0].str.lower()

domaines_d = set(dom(d["url_norm"]))
test_ext["domaine"] = dom(test_ext["url_norm"])

avant = len(test_ext)
test_ext = test_ext[~test_ext["domaine"].isin(domaines_d)].reset_index(drop=True)

print("Avant :", avant, "| Après :", len(test_ext), "| Retirées :", avant - len(test_ext))
print("Domaines communs restants :", len(set(test_ext["domaine"]) & domaines_d))

Avant : 54250 | Après : 48403 | Retirées : 5847
Domaines communs restants : 0


# 18-voir quels domaines étaient partagés

In [ ]:
d["domaine"] = dom(d["url_norm"])
d3_dom = dom(d3["url_norm"])

communs = set(d3_dom) & set(d["domaine"])

top = d3_dom[d3_dom.isin(communs)].value_counts().head(10)
tableau = pd.DataFrame({"URLs dans df3": top})
tableau["phishing dans d"] = [((d["domaine"] == x) & (d["is_phishing"] == 1)).sum() for x in top.index]
tableau["légitime dans d"] = [((d["domaine"] == x) & (d["is_phishing"] == 0)).sum() for x in top.index]
print(tableau)

                   URLs dans df3  phishing dans d  légitime dans d
url_norm                                                          
docs.google.com             3163               67                0
sites.google.com             656              976              144
tinyurl.com                  644              102                0
shorturl.at                  230                7                0
is.gd                        217                4                0
t.co                         195               63                1
bit.ly                       169              172                0
forms.office.com              73               31                0
script.google.com             73                5                0
s.free.fr                     41                1                0


# 19-réécrire extract_features

In [ ]:
MOTS = ["login", "signin", "verify", "account", "update", "secure",
        "bank", "confirm", "password", "paypal", "wallet"]
IP = re.compile(r"^\d{1,3}(?:\.\d{1,3}){3}$")

def extract_features(url):
    u = normaliser(url)                       # même nettoyage partout
    avant_q, _, query = u.partition("?")
    domaine, _, chemin = avant_q.partition("/")
    domaine = domaine.split(":")[0].lower()   # sans le port
    bas = u.lower()

    return {
        "url_length": len(u),
        "n_dots": u.count("."),
        "n_hyphens": u.count("-"),
        "n_underscores": u.count("_"),
        "n_slashes": u.count("/"),
        "n_at": u.count("@"),
        "n_question": u.count("?"),
        "n_equals": u.count("="),
        "n_ampersand": u.count("&"),
        "n_digits": sum(c.isdigit() for c in u),
        "n_letters": sum(c.isalpha() for c in u),
        "n_special": sum(not c.isalnum() for c in u),
        "has_ip": int(bool(IP.match(domaine))),
        "domain_length": len(domaine),
        "n_subdomains": 0 if IP.match(domaine) else max(domaine.count(".") - 1, 0),
        "path_length": len(chemin),
        "query_length": len(query),
        "n_suspicious_words": sum(bas.count(m) for m in MOTS),
    }

tests = ["example.com/login",
         "https://www.paypal.com.secure-update.xyz/verify",
         "192.168.0.1/index.php?id=1"]
print(pd.DataFrame([extract_features(t) for t in tests], index=["t1", "t2", "t3"]).T)

                    t1  t2  t3
url_length          17  35  26
n_dots               1   3   4
n_hyphens            0   1   0
n_underscores        0   0   0
n_slashes            1   1   1
n_at                 0   0   0
n_question           0   0   1
n_equals             0   0   1
n_ampersand          0   0   0
n_digits             0   0   9
n_letters           15  30  10
n_special            2   5   7
has_ip               0   0   1
domain_length       11  28  11
n_subdomains         0   2   0
path_length          5   6   9
query_length         0   0   4
n_suspicious_words   1   4   0


# 20-calculer les features sur d et test_ext

In [ ]:
import time
t0 = time.time()

X = pd.DataFrame([extract_features(u) for u in d["url_norm"]])
y = d["is_phishing"].values

X_ext = pd.DataFrame([extract_features(u) for u in test_ext["url_norm"]])
y_ext = test_ext["is_phishing"].values

print("X :", X.shape, "| y :", y.shape)
print("X_ext :", X_ext.shape, "| y_ext :", y_ext.shape)
print("Valeurs manquantes :", X.isna().sum().sum(), X_ext.isna().sum().sum())
print("Durée :", round(time.time() - t0), "s")

X : (443148, 18) | y : (443148,)
X_ext : (48403, 18) | y_ext : (48403,)
Valeurs manquantes : 0 0
Durée : 13 s


# 21-split train/test par domaine

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

groupes = d["domaine"].values   # un groupe = un domaine

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
idx_train, idx_test = next(gss.split(X, y, groups=groupes))

X_train, X_test = X.iloc[idx_train], X.iloc[idx_test]
y_train, y_test = y[idx_train], y[idx_test]

print("Train :", X_train.shape, "| Test :", X_test.shape)
print("% phishing train :", round(y_train.mean() * 100, 1))
print("% phishing test  :", round(y_test.mean() * 100, 1))

dom_train = set(groupes[idx_train])
dom_test = set(groupes[idx_test])
print("Domaines communs train/test :", len(dom_train & dom_test))

Train : (359022, 18) | Test : (84126, 18)
% phishing train : 21.8
% phishing test  : 22.6
Domaines communs train/test : 0


# 22-modèle de référence, Logistic Regression

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

lr = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
)
lr.fit(X_train, y_train)

proba = lr.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print(classification_report(y_test, pred, target_names=["légitime", "phishing"], digits=3))
print("ROC-AUC :", round(roc_auc_score(y_test, proba), 3))
print("Matrice de confusion (lignes = réel, colonnes = prédit) :")
print(confusion_matrix(y_test, pred))

pred_ext = lr.predict(X_ext)
print("Rappel sur le test externe :", round(pred_ext.mean(), 3))

              precision    recall  f1-score   support

    légitime      0.883     0.765     0.820     65087
    phishing      0.448     0.653     0.532     19039

    accuracy                          0.740     84126
   macro avg      0.666     0.709     0.676     84126
weighted avg      0.785     0.740     0.755     84126

ROC-AUC : 0.8
Matrice de confusion (lignes = réel, colonnes = prédit) :
[[49790 15297]
 [ 6606 12433]]
Rappel sur le test externe : 0.511


# 23-: Decision Tree et Random Forest

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score

modeles = {
    "Decision Tree": DecisionTreeClassifier(max_depth=12, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, min_samples_leaf=5,
                                            class_weight="balanced", n_jobs=-1, random_state=42),
}

lignes = []
for nom, m in modeles.items():
    m.fit(X_train, y_train)
    p = m.predict_proba(X_test)[:, 1]
    pr = (p >= 0.5).astype(int)
    lignes.append({
        "modèle": nom,
        "precision": precision_score(y_test, pr),
        "recall": recall_score(y_test, pr),
        "F1": f1_score(y_test, pr),
        "ROC-AUC": roc_auc_score(y_test, p),
        "rappel externe": m.predict(X_ext).mean(),
    })

print(pd.DataFrame(lignes).round(3).to_string(index=False))

       modèle  precision  recall    F1  ROC-AUC  rappel externe
Decision Tree      0.514   0.800 0.626    0.873           0.693
Random Forest      0.585   0.847 0.692    0.916           0.721


# 24-n-grammes de caractères (TF-IDF) + régression logistique

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3, 5),
                        max_features=200000, sublinear_tf=True, lowercase=True)

txt_train = d["url_norm"].values[idx_train]
txt_test = d["url_norm"].values[idx_test]

Xt_train = tfidf.fit_transform(txt_train)       # appris sur le train seulement
Xt_test = tfidf.transform(txt_test)
Xt_ext = tfidf.transform(test_ext["url_norm"].values)

clf = LogisticRegression(max_iter=300, class_weight="balanced", solver="saga", random_state=42)
clf.fit(Xt_train, y_train)

p = clf.predict_proba(Xt_test)[:, 1]
pr = (p >= 0.5).astype(int)
print("precision :", round(precision_score(y_test, pr), 3))
print("recall    :", round(recall_score(y_test, pr), 3))
print("F1        :", round(f1_score(y_test, pr), 3))
print("ROC-AUC   :", round(roc_auc_score(y_test, p), 3))
print("rappel externe :", round(clf.predict(Xt_ext).mean(), 3))

precision : 0.83
recall    : 0.942
F1        : 0.882
ROC-AUC   : 0.986
rappel externe : 0.906


# 25-regarder ce que le modèle a appris

In [ ]:
noms = tfidf.get_feature_names_out()
coef = clf.coef_[0]
ordre = coef.argsort()

print("Top 25 n-grammes → PHISHING")
for i in ordre[::-1][:25]:
    print(f"{coef[i]:6.2f}  {noms[i]!r}")

print("\nTop 25 n-grammes → LÉGITIME")
for i in ordre[:25]:
    print(f"{coef[i]:6.2f}  {noms[i]!r}")

Top 25 n-grammes → PHISHING
  7.95  '...'
  6.61  'doc'
  6.49  'php'
  6.31  '.exe'
  6.29  'mail'
  6.27  '&am'
  6.18  '&amp'
  5.94  '.ru/'
  5.90  'ru/'
  5.83  '.ex'
  5.75  '/gd'
  5.61  'exe'
  5.53  '.php'
  5.27  '.ph'
  5.23  '/log'
  5.16  '/im'
  5.15  'mxp'
  4.97  '/js'
  4.90  '/wp'
  4.80  '/up'
  4.74  'mai'
  4.69  'ogin'
  4.64  'login'
  4.62  'wp-'
  4.61  'ss/'

Top 25 n-grammes → LÉGITIME
 -7.47  'htm'
 -6.73  '.com/'
 -6.50  'com/'
 -6.38  'om/'
 -6.33  '.htm'
 -6.03  '.edu/'
 -6.00  '.ht'
 -5.85  'film'
 -5.82  '.com'
 -5.78  '.asp'
 -5.66  'edu/'
 -5.64  'du/'
 -5.40  '200'
 -5.14  '.edu'
 -5.09  'ilm'
 -5.02  '.as'
 -4.96  'com'
 -4.95  '.co'
 -4.95  '2011'
 -4.89  'wiki'
 -4.77  'wik'
 -4.69  'ca/'
 -4.63  '.ed'
 -4.59  '.ca/'
 -4.58  'id='


# 26-vérifier l'artefact

In [ ]:
masque = d["url_norm"].str.contains(r"\.\.\.", regex=True)
print("URLs avec '...' :", masque.sum())
print(d.loc[masque, "is_phishing"].value_counts())
print("\nParmi test_ext :", test_ext["url_norm"].str.contains(r"\.\.\.", regex=True).sum())

print("\nExemples (80 premiers caractères) :")
for u, y_ in d.loc[masque, ["url_norm", "is_phishing"]].head(8).values:
    print(y_, repr(u[:80]))

URLs avec '...' : 2672
is_phishing
1    2525
0     147
Name: count, dtype: int64

Parmi test_ext : 17

Exemples (80 premiers caractères) :
0 'downloads.khinsider.com/game-soundtracks/album/ragnarok-the-animation-original-s'
0 'en.wikipedia.org/wiki/...Baby_One_More_Time_(song)'
0 'en.wikipedia.org/wiki/...Sans_laisser_d%27adresse'
0 'en.wikipedia.org/wiki/All_Good_Things..._%28TNG_episode%29'
0 'en.wikipedia.org/wiki/I_Am..._Sasha_Fierce'
0 'en.wikipedia.org/wiki/Let%27s_Die_Forever..._Together'
0 'itunes.apple.com/ca/album/1992-2002-tout.../id330322250'
0 'itunes.apple.com/us/album/taste-of-rain...-why-kneel/id272909944'


# 27-retirer ... et ré-entraîner le TF-IDF

In [ ]:
def nettoyer_txt(u):
    return re.sub(r"\.{3,}", "", u)       # retire les suites de 3 points ou plus

d["txt"] = d["url_norm"].apply(nettoyer_txt)
test_ext["txt"] = test_ext["url_norm"].apply(nettoyer_txt)

tfidf = TfidfVectorizer(analyzer="char", ngram_range=(3, 5),
                        max_features=200000, sublinear_tf=True, lowercase=True)

Xt_train = tfidf.fit_transform(d["txt"].values[idx_train])
Xt_test = tfidf.transform(d["txt"].values[idx_test])
Xt_ext = tfidf.transform(test_ext["txt"].values)

clf = LogisticRegression(max_iter=300, class_weight="balanced", solver="saga", random_state=42)
clf.fit(Xt_train, y_train)

p = clf.predict_proba(Xt_test)[:, 1]
pr = (p >= 0.5).astype(int)
print("precision :", round(precision_score(y_test, pr), 3))
print("recall    :", round(recall_score(y_test, pr), 3))
print("F1        :", round(f1_score(y_test, pr), 3))
print("ROC-AUC   :", round(roc_auc_score(y_test, p), 3))
print("rappel externe :", round(clf.predict(Xt_ext).mean(), 3))

precision : 0.829
recall    : 0.942
F1        : 0.882
ROC-AUC   : 0.986
rappel externe : 0.907


# 28-retirer le TLD et les extensions de page

In [ ]:
def retirer_tld_ext(u):
    u = nettoyer_txt(u)
    m = re.match(r"([^/?#]*)(.*)", u)
    dom, reste = m.group(1), m.group(2)
    # domaine : on retire le dernier suffixe (.com, .ru...) sauf si c'est une IP
    if not re.fullmatch(r"[\d.:]+", dom):
        dom = re.sub(r"\.[a-zA-Z]{2,}(?=(:\d+)?$)", "", dom)
    # chemin : on retire les extensions de page web classiques
    reste = re.sub(r"\.(html?|shtml|aspx?|jsp|cfm)(?=$|[?#])", "", reste, flags=re.I)
    return dom + reste

d["txt2"] = d["url_norm"].apply(retirer_tld_ext)
test_ext["txt2"] = test_ext["url_norm"].apply(retirer_tld_ext)

# aperçu (affiché comme texte, jamais ouvert)
for i in [0, 1, 2]:
    print(d["txt"].iloc[i], "->", d["txt2"].iloc[i])

tfidf2 = TfidfVectorizer(analyzer="char", ngram_range=(3, 5),
                         max_features=200000, sublinear_tf=True, lowercase=True)
Xt_train2 = tfidf2.fit_transform(d["txt2"].values[idx_train])
Xt_test2 = tfidf2.transform(d["txt2"].values[idx_test])
Xt_ext2 = tfidf2.transform(test_ext["txt2"].values)

clf2 = LogisticRegression(max_iter=300, class_weight="balanced", solver="saga", random_state=42)
clf2.fit(Xt_train2, y_train)

p2 = clf2.predict_proba(Xt_test2)[:, 1]
pr2 = (p2 >= 0.5).astype(int)
print("precision :", round(precision_score(y_test, pr2), 3))
print("recall    :", round(recall_score(y_test, pr2), 3))
print("F1        :", round(f1_score(y_test, pr2), 3))
print("ROC-AUC   :", round(roc_auc_score(y_test, p2), 3))
print("rappel externe :", round(clf2.predict(Xt_ext2).mean(), 3))

google.com -> google
youtube.com -> youtube
facebook.com -> facebook
precision : 0.806
recall    : 0.926
F1        : 0.862
ROC-AUC   : 0.981
rappel externe : 0.848


# 29-combiner n-grammes et 18 features

In [ ]:
import numpy as np
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler

# comptages : log1p pour calmer les valeurs extrêmes, puis standardisation
sc = StandardScaler()
F_train = sc.fit_transform(np.log1p(X_train))
F_test = sc.transform(np.log1p(X_test))
F_ext = sc.transform(np.log1p(X_ext))

# on colle les features aux n-grammes (TF-IDF de txt2)
Xc_train = hstack([Xt_train2, csr_matrix(F_train)]).tocsr()
Xc_test = hstack([Xt_test2, csr_matrix(F_test)]).tocsr()
Xc_ext = hstack([Xt_ext2, csr_matrix(F_ext)]).tocsr()

clf3 = LogisticRegression(max_iter=300, class_weight="balanced", solver="saga", random_state=42)
clf3.fit(Xc_train, y_train)

p3 = clf3.predict_proba(Xc_test)[:, 1]
pr3 = (p3 >= 0.5).astype(int)
print("precision :", round(precision_score(y_test, pr3), 3))
print("recall    :", round(recall_score(y_test, pr3), 3))
print("F1        :", round(f1_score(y_test, pr3), 3))
print("ROC-AUC   :", round(roc_auc_score(y_test, p3), 3))
print("rappel externe :", round(clf3.predict(Xc_ext).mean(), 3))

precision : 0.73
recall    : 0.924
F1        : 0.816
ROC-AUC   : 0.976
rappel externe : 0.805


/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


# 30-LightGBM sur les 18 features seules

In [ ]:
import lightgbm as lgb

gbm = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.1, num_leaves=31,
                         class_weight="balanced", random_state=42, verbose=-1)
gbm.fit(X_train, y_train)

p4 = gbm.predict_proba(X_test)[:, 1]
pr4 = (p4 >= 0.5).astype(int)
print("precision :", round(precision_score(y_test, pr4), 3))
print("recall    :", round(recall_score(y_test, pr4), 3))
print("F1        :", round(f1_score(y_test, pr4), 3))
print("ROC-AUC   :", round(roc_auc_score(y_test, p4), 3))
print("rappel externe :", round(gbm.predict(X_ext).mean(), 3))

precision : 0.578
recall    : 0.854
F1        : 0.69
ROC-AUC   : 0.916
rappel externe : 0.728


# 31-empiler (stacking léger) TF-IDF + comptages

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict

groupes = d["domaine"].values[idx_train]
gkf = GroupKFold(n_splits=3)

# probabilité TF-IDF out-of-fold sur le train (chaque ligne prédite par un modèle qui ne l'a pas vue)
clf_oof = LogisticRegression(max_iter=300, class_weight="balanced", solver="saga", random_state=42)
p_oof = cross_val_predict(clf_oof, Xt_train2, y_train, cv=gkf, groups=groupes,
                          method="predict_proba")[:, 1]

# probabilités TF-IDF pour test et externe : modèle clf2 (cellule 29)
p_te = clf2.predict_proba(Xt_test2)[:, 1]
p_ex = clf2.predict_proba(Xt_ext2)[:, 1]

# LightGBM sur 18 comptages + probabilité TF-IDF
S_train = np.column_stack([X_train.values, p_oof])
S_test = np.column_stack([X_test.values, p_te])
S_ext = np.column_stack([X_ext.values, p_ex])

gbm2 = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31,
                          class_weight="balanced", random_state=42, verbose=-1)
gbm2.fit(S_train, y_train)

p5 = gbm2.predict_proba(S_test)[:, 1]
pr5 = (p5 >= 0.5).astype(int)
print("precision :", round(precision_score(y_test, pr5), 3))
print("recall    :", round(recall_score(y_test, pr5), 3))
print("F1        :", round(f1_score(y_test, pr5), 3))
print("ROC-AUC   :", round(roc_auc_score(y_test, p5), 3))
print("rappel externe :", round(gbm2.predict(S_ext).mean(), 3))

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


precision : 0.838
recall    : 0.95
F1        : 0.891
ROC-AUC   : 0.988


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


rappel externe : 0.877


# 32-choisir le seuil

In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, seuils = precision_recall_curve(y_test, p5)

for s in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred = (p5 >= s).astype(int)
    fp = int(((pred == 1) & (y_test == 0)).sum())
    fn = int(((pred == 0) & (y_test == 1)).sum())
    print(f"seuil {s} | precision {precision_score(y_test, pred):.3f} | recall {recall_score(y_test, pred):.3f} | faux positifs {fp} | faux négatifs {fn} | rappel externe {(gbm2.predict_proba(S_ext)[:, 1] >= s).mean():.3f}")

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


seuil 0.3 | precision 0.766 | recall 0.966 | faux positifs 5630 | faux négatifs 644 | rappel externe 0.926


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


seuil 0.4 | precision 0.804 | recall 0.958 | faux positifs 4435 | faux négatifs 804 | rappel externe 0.901


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

seuil 0.7 | precision 0.890 | recall 0.928 | faux positifs 2194 | faux négatifs 1362 | rappel externe 0.825


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


seuil 0.8 | precision 0.916 | recall 0.911 | faux positifs 1591 | faux négatifs 1695 | rappel externe 0.787


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


seuil 0.9 | precision 0.950 | recall 0.878 | faux positifs 886 | faux négatifs 2327 | rappel externe 0.718


# 33-sauvegarder le modèle

In [ ]:
import joblib

SEUIL = 0.6
paquet = {"tfidf": tfidf2, "clf_tfidf": clf2, "gbm": gbm2,
          "seuil": SEUIL, "colonnes": list(X_train.columns)}
joblib.dump(paquet, "detecteur_phishing.joblib")

import os
print("taille (Mo) :", round(os.path.getsize("detecteur_phishing.joblib") / 1e6, 1))
print("colonnes :", paquet["colonnes"])

taille (Mo) : 9.5
colonnes : ['url_length', 'n_dots', 'n_hyphens', 'n_underscores', 'n_slashes', 'n_at', 'n_question', 'n_equals', 'n_ampersand', 'n_digits', 'n_letters', 'n_special', 'has_ip', 'domain_length', 'n_subdomains', 'path_length', 'query_length', 'n_suspicious_words']


# 34-fonction de prédiction avec input()

In [ ]:
import joblib
import pandas as pd

paquet = joblib.load("detecteur_phishing.joblib")
tfidf_p, clf_p, gbm_p = paquet["tfidf"], paquet["clf_tfidf"], paquet["gbm"]
seuil_p, colonnes_p = paquet["seuil"], paquet["colonnes"]

def predire_url(url):
    # 1) même chaîne de traitement qu'à l'entraînement
    u = normaliser(url)
    txt2 = retirer_tld_ext(u)
    # 2) probabilité du modèle TF-IDF
    p_tfidf = clf_p.predict_proba(tfidf_p.transform([txt2]))[:, 1][0]
    # 3) 18 comptages + probabilité TF-IDF -> LightGBM
    feats = extract_features(url)
    ligne = pd.DataFrame([feats])[colonnes_p]
    ligne["p_tfidf"] = p_tfidf
    proba = gbm_p.predict_proba(ligne.values)[:, 1][0]
    # 4) décision au seuil choisi
    if proba >= seuil_p:
        print(f"⚠️ PHISHING  (probabilité de phishing : {proba*100:.1f} %)")
    else:
        print(f"✅ URL LÉGITIME  (probabilité de phishing : {proba*100:.1f} %)")
    print("Aide à la décision, pas un verdict.")

# test sur 3 URLs (affichées comme texte, jamais ouvertes)
for t in ["google.com", "youtube.com/watch?v=abc", "secure-login-paypal.verify-account.example.tk/update.php"]:
    print(t)
    predire_url(t)
    print()

url = input("Entrez une URL : ")
predire_url(url)

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


google.com
✅ URL LÉGITIME  (probabilité de phishing : 39.5 %)
Aide à la décision, pas un verdict.

youtube.com/watch?v=abc
✅ URL LÉGITIME  (probabilité de phishing : 0.1 %)
Aide à la décision, pas un verdict.

secure-login-paypal.verify-account.example.tk/update.php
⚠️ PHISHING  (probabilité de phishing : 99.9 %)
Aide à la décision, pas un verdict.

Entrez une URL : https://account-verification.example/office365/signin
⚠️ PHISHING  (probabilité de phishing : 100.0 %)
Aide à la décision, pas un verdict.


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# 35-tester sur des URLs légitimes avec mots sensibles et sans mots-clés

In [ ]:
tests = [
    ("légitime, domaine nu",            "wikipedia.org"),
    ("légitime, domaine nu",            "github.com"),
    ("légitime, avec chemin",           "github.com/scikit-learn/scikit-learn"),
    ("légitime, mot sensible",          "accounts.google.com/signin"),
    ("légitime, mot sensible",          "paypal.com/signin"),
    ("légitime, mot sensible",          "login.microsoftonline.com"),
    ("phishing fictif, sans mot-clé",   "kx7-promo-gagnant.example.xyz/offre/12345"),
    ("phishing fictif, IP",             "192.0.2.10/mon-compte/index.php"),
    ("phishing fictif, imitation",      "paypa1-service.example.com/secure"),
]
for etiquette, t in tests:
    print(etiquette, "|", t)
    predire_url(t)
    print()

légitime, domaine nu | wikipedia.org
✅ URL LÉGITIME  (probabilité de phishing : 0.4 %)
Aide à la décision, pas un verdict.

légitime, domaine nu | github.com
✅ URL LÉGITIME  (probabilité de phishing : 55.6 %)
Aide à la décision, pas un verdict.

légitime, avec chemin | github.com/scikit-learn/scikit-learn
✅ URL LÉGITIME  (probabilité de phishing : 2.2 %)
Aide à la décision, pas un verdict.

légitime, mot sensible | accounts.google.com/signin
⚠️ PHISHING  (probabilité de phishing : 99.9 %)
Aide à la décision, pas un verdict.

légitime, mot sensible | paypal.com/signin
⚠️ PHISHING  (probabilité de phishing : 97.1 %)
Aide à la décision, pas un verdict.

légitime, mot sensible | login.microsoftonline.com
⚠️ PHISHING  (probabilité de phishing : 99.4 %)
Aide à la décision, pas un verdict.

phishing fictif, sans mot-clé | kx7-promo-gagnant.example.xyz/offre/12345
⚠️ PHISHING  (probabilité de phishing : 99.3 %)
Aide à la décision, pas un verdict.

phishing fictif, IP | 192.0.2.10/mon-compte/in

/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/ut

# 36-mesurer l’hypothèse « mots sensibles = phishing » dans d

In [ ]:
mots = ["login", "signin", "account", "verify", "secure", "update", "paypal"]
for m in mots:
    masque = d["url_norm"].str.lower().str.contains(m, regex=False)
    n = int(masque.sum())
    if n > 0:
        part_phish = d.loc[masque, "is_phishing"].mean()
        print(f"{m:8s} | URLs : {n:6d} | part de phishing : {part_phish*100:5.1f} %")
    else:
        print(f"{m:8s} | aucune URL")
print("référence : part de phishing globale =", round(d["is_phishing"].mean()*100, 1), "%")

login    | URLs :   9440 | part de phishing :  99.0 %
signin   | URLs :   1019 | part de phishing :  95.2 %
account  | URLs :   2592 | part de phishing :  92.2 %
verify   | URLs :    745 | part de phishing :  99.7 %
secure   | URLs :   3376 | part de phishing :  97.5 %
update   | URLs :   2784 | part de phishing :  89.8 %
paypal   | URLs :   1650 | part de phishing :  99.9 %
référence : part de phishing globale = 22.0 %


# 37-regarder les légitimes qui contiennent ces mots

In [ ]:
masque = d["url_norm"].str.lower().str.contains("login", regex=False) & (d["is_phishing"] == 0)
print("légitimes contenant 'login' :", int(masque.sum()))
print()
for u in d.loc[masque, "url_norm"].head(15):
    print(u[:100])

légitimes contenant 'login' : 92

login.tmall.com
onelogin.com
asmt.mafoi.com/login.cfm?mftypeofuser=M
bluegrasspolitics.bloginky.com/2009/12/16/veteran-state-officeholder-drex-davis-dies-at-88/
bookcritics.org/blog/archive/roundup-jonathan-lethem-hector-tobar-john-jeremiah-sullivan-clogins-and
daymix.com/Pandora-Radio-Login/
dscoop.saepio.com/ams/urlauthlogin.do?user=-username-
email12.secureserver.net/webmail.php?login=1
etrakit.capecoral.net/etrakit2/Login.aspx?type=contractor
girlfashionmall.com/login.php
home.intair.com/authentification/Login/Welcome.php?action=NWB
investments.cwbkc.com/src/Login/Login.aspx
jobsandsafecommunities.com/wp-content/plugins/wp-password/login.php?err=&destination=%2F
lineup.osheaga.com/login
loomis.saepio.com/ams/login.do?s=DQ


# 38-liste blanche de domaines de confiance

In [ ]:
DOMAINES_CONFIANCE = {
    "google.com", "accounts.google.com", "paypal.com", "microsoft.com",
    "login.microsoftonline.com", "github.com", "wikipedia.org",
    "youtube.com", "facebook.com", "apple.com", "amazon.com",
}

def hote(url):
    u = normaliser(url)
    return re.match(r"[^/?#:]*", u).group(0).lower()

def predire_url2(url):
    h = hote(url)
    if h in DOMAINES_CONFIANCE:
        print(f"✅ URL LÉGITIME  (domaine de la liste blanche : {h})")
        print("Décision par liste blanche, pas par le modèle.")
        return
    predire_url(url)

for t in ["accounts.google.com/signin", "paypal.com/signin", "login.microsoftonline.com",
          "accounts.google.com.verif-compte.example.tk/signin", "paypa1-service.example.com/secure"]:
    print(t)
    predire_url2(t)
    print()

accounts.google.com/signin
✅ URL LÉGITIME  (domaine de la liste blanche : accounts.google.com)
Décision par liste blanche, pas par le modèle.

paypal.com/signin
✅ URL LÉGITIME  (domaine de la liste blanche : paypal.com)
Décision par liste blanche, pas par le modèle.

login.microsoftonline.com
✅ URL LÉGITIME  (domaine de la liste blanche : login.microsoftonline.com)
Décision par liste blanche, pas par le modèle.

accounts.google.com.verif-compte.example.tk/signin
⚠️ PHISHING  (probabilité de phishing : 100.0 %)
Aide à la décision, pas un verdict.

paypa1-service.example.com/secure
⚠️ PHISHING  (probabilité de phishing : 99.9 %)
Aide à la décision, pas un verdict.



/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


# 39-interface Gradio

In [ ]:
!pip install -q gradio
import gradio as gr

def analyser(url):
    url = (url or "").strip()
    if not url:
        return "Entrez une URL."
    h = hote(url)
    if h in DOMAINES_CONFIANCE:
        return f"✅ URL LÉGITIME\nDécision : liste blanche ({h})\nAide à la décision, pas un verdict."
    u = normaliser(url)
    txt2 = retirer_tld_ext(u)
    p_tfidf = clf_p.predict_proba(tfidf_p.transform([txt2]))[:, 1][0]
    ligne = pd.DataFrame([extract_features(url)])[colonnes_p]
    ligne["p_tfidf"] = p_tfidf
    proba = gbm_p.predict_proba(ligne.values)[:, 1][0]
    verdict = "⚠️ PHISHING" if proba >= seuil_p else "✅ URL LÉGITIME"
    return f"{verdict}\nProbabilité de phishing : {proba*100:.1f} %\nDécision : modèle (seuil {seuil_p})\nAide à la décision, pas un verdict."

demo = gr.Interface(fn=analyser,
                    inputs=gr.Textbox(label="URL à analyser (texte uniquement, jamais ouverte)"),
                    outputs=gr.Textbox(label="Résultat"),
                    title="Détecteur de phishing par URL",
                    description="Projet ML : analyse du texte de l'URL uniquement. Outil pédagogique, pas un verdict.")
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://53c12418e392c4c626.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# 40-tableau récapitulatif final

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score

SEUIL = 0.6
pred = (p5 >= SEUIL).astype(int)

print("matrice de confusion [[VN, FP], [FN, VP]] :")
print(confusion_matrix(y_test, pred))
print()
print("accuracy  :", round(accuracy_score(y_test, pred), 3))
print("precision :", round(precision_score(y_test, pred), 3))
print("recall    :", round(recall_score(y_test, pred), 3))
print("F1        :", round(f1_score(y_test, pred), 3))
print("ROC-AUC   :", round(roc_auc_score(y_test, p5), 3))
print("rappel externe :", round((gbm2.predict_proba(S_ext)[:, 1] >= SEUIL).mean(), 3))
print()
print("accuracy si on prédit toujours 'légitime' :", round(1 - y_test.mean(), 3))

matrice de confusion [[VN, FP], [FN, VP]] :
[[62316  2771]
 [ 1127 17912]]

accuracy  : 0.954
precision : 0.866
recall    : 0.941
F1        : 0.902
ROC-AUC   : 0.988


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


rappel externe : 0.853

accuracy si on prédit toujours 'légitime' : 0.774
